# FireSight-IR | Module 1a — VIIRS & MODIS Cloud-Native Ingestion

**Project:** FireSight-IR — Physics-constrained wildfire intelligence at constellation scale  
**Author:** Emmanuel Ibekwe  
**GitHub:** github.com/Ibekwemmanuel7/firesight-ir  

---

## Objectives

This notebook is the entry point of the FireSight-IR data pipeline. It performs **cloud-native** ingestion of VIIRS and MODIS thermal infrared granules from NASA Earthdata S3 — no files are downloaded to local disk. All data is streamed into memory, processed, and written to a structured HDF5 patch archive.

### What this notebook does
1. **Authenticate** to NASA Earthdata via `earthaccess` (OAuth2 + S3 credential management)
2. **Define** the Western US study region and 2018–2022 temporal window
3. **Query** VIIRS VNP14IMG (I4/I5 thermal anomaly) and MODIS MOD14/MYD14 granules via CMR-STAC
4. **Stream** granule data directly from NASA S3 (us-west-2) into memory using `xarray` + `h5py`
5. **Extract** brightness temperatures (BT_I4, BT_I5) and candidate thermal anomaly pixels
6. **Write** a structured HDF5 patch archive: one 32×32 patch per candidate pixel

### Data sources used here
| Dataset | Short name | Resolution | Variable |
|---|---|---|---|
| VIIRS I-band thermal anomaly | VNP14IMG | 375 m | BrightT4, BrightT5, fire mask |
| MODIS Terra fire | MOD14 | 1 km | fire mask, FP_power |
| MODIS Aqua fire | MYD14 | 1 km | fire mask, FP_power |

> **FireSat alignment note:** VIIRS I4 (3.7 µm) and I5 (11.0 µm) are the closest available analogs
> to FireSat's MWIR/LWIR channels at 375 m resolution. The BT differential (BT_I4 − BT_I5)
> is the primary fire detection signal throughout this pipeline.

---

## Environment Setup

Install once per environment. If using Anaconda on Windows, run the conda command first,
then pip for earthaccess:

```
conda install -c conda-forge xarray h5py h5netcdf rasterio shapely tqdm -y
pip install earthaccess
```

In [1]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import io
import json
import datetime
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import h5py
import xarray as xr
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from shapely.geometry import box

# NASA Earthdata cloud access
import earthaccess

warnings.filterwarnings('ignore', category=RuntimeWarning)
print("✓ Imports complete")

✓ Imports complete


---
## 1. Configuration — Study Region, Time Window, Paths

In [2]:
# ── Study region: Western US (CA + NV + OR) ───────────────────────────────────
# Bounding box: (lon_min, lat_min, lon_max, lat_max)
# Covers California, Nevada, and Oregon
BBOX = (-124.5, 32.5, -114.0, 46.5)  # lon_min, lat_min, lon_max, lat_max
BBOX_SHAPELY = box(*BBOX)             # shapely polygon for spatial ops

# ── Temporal window: 2018–2022 ────────────────────────────────────────────────
# We will query year-by-year to manage memory and allow checkpointing
YEARS = list(range(2018, 2023))       # [2018, 2019, 2020, 2021, 2022]
DATE_FORMAT = "%Y-%m-%d"

# Annual fire season window (peak months): June 1 – November 30
# This captures ~85% of Western US large fire events while reducing data volume ~2×
SEASON_START_MMDD = "06-01"
SEASON_END_MMDD   = "11-30"

# ── Patch parameters ──────────────────────────────────────────────────────────
PATCH_SIZE = 32           # pixels (32×32 = ~12 km × 12 km at 375 m)
BT_I4_THRESHOLD = 310.0  # K — adaptive lower bound for thermal anomaly candidates
                          # Will be refined per-scene using local background stats

# ── Output paths ─────────────────────────────────────────────────────────────
# All outputs go into a local 'data/' directory — the HDF5 is the only disk write
DATA_DIR    = Path("data")
ARCHIVE_DIR = DATA_DIR / "patches"
CATALOG_DIR = DATA_DIR / "catalogs"
FIGURES_DIR = Path("figures")

for d in [ARCHIVE_DIR, CATALOG_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Study region:  BBOX = {BBOX}")
print(f"✓ Temporal window: {YEARS[0]}–{YEARS[-1]}, fire season only")
print(f"✓ Patch size: {PATCH_SIZE}×{PATCH_SIZE} pixels")
print(f"✓ Output directory: {ARCHIVE_DIR.resolve()}")

✓ Study region:  BBOX = (-124.5, 32.5, -114.0, 46.5)
✓ Temporal window: 2018–2022, fire season only
✓ Patch size: 32×32 pixels
✓ Output directory: C:\Users\taylo\firesat\data\patches


---
## 2. NASA Earthdata Authentication

`earthaccess` handles OAuth2 + temporary S3 credentials automatically.  
You need a free NASA Earthdata account: https://urs.earthdata.nasa.gov/

On first run, `earthaccess.login(strategy='interactive')` will prompt for username/password
and save credentials to `~/.netrc` for future sessions. On subsequent runs use `strategy='netrc'`.

In [3]:
# ── Authenticate ──────────────────────────────────────────────────────────────
# First run:       strategy='interactive'  (prompts for credentials, saves to ~/.netrc)
# Subsequent runs: strategy='netrc'        (reads saved credentials silently)

auth = earthaccess.login(strategy='interactive')

# Verify we are authenticated and can get S3 credentials
assert auth.authenticated, "Authentication failed — check your NASA Earthdata credentials."
print(f"✓ Authenticated as: {auth.username}")
print("✓ S3 direct access enabled — no downloads to local disk")

Enter your Earthdata Login username:  Msledge7
Enter your Earthdata password:  ········


✓ Authenticated as: Msledge7
✓ S3 direct access enabled — no downloads to local disk


In [4]:
import earthaccess
print(earthaccess.__version__)

0.15.1


---
## 3. CMR Granule Search

We query the NASA Common Metadata Repository (CMR) for three collections:

| Collection | Short name | Provider | Description |
|---|---|---|---|
| VIIRS I-band fire | `VNP14IMG` | LPDAAC | 375 m thermal anomaly, BT_I4, BT_I5 |
| MODIS Terra fire | `MOD14` | LPDAAC | 1 km fire radiative power |
| MODIS Aqua fire | `MYD14` | LPDAAC | 1 km fire radiative power |

Searches are scoped to the Western US bounding box and fire-season months only.

In [ ]:
# ── Collection short names ─────────────────────────────────────────────────────
COLLECTIONS = {
    "VNP14IMG": {"short_name": "VNP14IMG",   "version": "001", "description": "VIIRS 375m thermal anomaly"},
    "MOD14":    {"short_name": "MOD14",       "version": "061", "description": "MODIS Terra 1km fire"},
    "MYD14":    {"short_name": "MYD14",       "version": "061", "description": "MODIS Aqua 1km fire"},
}

def query_granules_for_year(short_name: str, version: str, year: int,
                             bbox: tuple, season_start: str, season_end: str,
                             max_results: int = 5000) -> list:
    """
    Query NASA CMR for granules matching a collection, year, and bounding box.
    Returns a list of earthaccess DataGranule objects.
    
    Parameters
    ----------
    short_name    : CMR collection short name (e.g. 'VNP14IMG')
    version       : collection version string (e.g. '001')
    year          : calendar year (e.g. 2020)
    bbox          : (lon_min, lat_min, lon_max, lat_max)
    season_start  : 'MM-DD' string for season start (e.g. '06-01')
    season_end    : 'MM-DD' string for season end   (e.g. '11-30')
    max_results   : CMR result cap per query
    
    Returns
    -------
    list of earthaccess DataGranule
    """
    date_start = f"{year}-{season_start}"
    date_end   = f"{year}-{season_end}"
    
    results = earthaccess.search_data(
        short_name    = short_name,
        version       = version,
        temporal      = (date_start, date_end),
        bounding_box  = bbox,
        count         = max_results,
    )
    return results


# ── Run queries for all collections and all years ─────────────────────────────
# Results are stored in a nested dict: granule_catalog[collection][year] = [granules]
granule_catalog = {name: {} for name in COLLECTIONS}

for coll_name, coll_info in COLLECTIONS.items():
    print(f"\n── Querying {coll_name} ({coll_info['description']}) ──")
    total = 0
    for year in YEARS:
        granules = query_granules_for_year(
            short_name   = coll_info["short_name"],
            version      = coll_info["version"],
            year         = year,
            bbox         = BBOX,
            season_start = SEASON_START_MMDD,
            season_end   = SEASON_END_MMDD,
        )
        granule_catalog[coll_name][year] = granules
        print(f"  {year}: {len(granules):>5} granules")
        total += len(granules)
    print(f"  TOTAL: {total} granules across 2018–2022")

# Save catalog summary to JSON for reference
catalog_summary = {
    coll: {str(yr): len(grans) for yr, grans in by_year.items()}
    for coll, by_year in granule_catalog.items()
}
with open(CATALOG_DIR / "granule_counts.json", "w") as f:
    json.dump(catalog_summary, f, indent=2)

print(f"\n✓ Catalog saved to {CATALOG_DIR / 'granule_counts.json'}")

In [ ]:
# ── Collection short names (updated for Earthdata Cloud) ─────────────────────
COLLECTIONS = {
    "VNP14IMG": {
        "short_name": "VNP14IMG",
        "version": "002",
        "description": "VIIRS 375m active fire (Suomi-NPP)"
    },
    "MOD14": {
        "short_name": "MOD14",
        "version": "061",
        "description": "MODIS Terra 1km fire"
    },
    "MYD14": {
        "short_name": "MYD14",
        "version": "061",
        "description": "MODIS Aqua 1km fire"
    }
}

---
## 4. Inspect a Single VIIRS Granule (Structure Audit)

Before processing thousands of granules, we open one granule directly from S3
to understand its HDF5 group/dataset structure. This is the equivalent of a
manual `ncdump -h` — essential before writing the batch processor.

In [ ]:
def query_granules_for_year(
        short_name: str,
        version: str,
        year: int,
        bbox: tuple,
        season_start: str,
        season_end: str,
        max_results: int = 5000
    ):
    """
    Query NASA CMR for granules matching a collection, year, and bounding box.
    """

    date_start = f"{year}-{season_start}"
    date_end   = f"{year}-{season_end}"

    results = earthaccess.search_data(
        short_name=short_name,
        version=version,
        temporal=(date_start, date_end),
        bounding_box=bbox,
        cloud_hosted=True,      # ensures S3 datasets
        count=max_results
    )

    return results

In [ ]:
def walk_hdf5_structure(h5file, prefix="", max_depth=4, _depth=0):
    """Recursively print HDF5 group/dataset names and shapes."""
    if _depth >= max_depth:
        return
    for key in h5file.keys():
        item = h5file[key]
        path = f"{prefix}/{key}"
        if isinstance(item, h5py.Dataset):
            print(f"  DATASET  {path:<60} shape={item.shape}  dtype={item.dtype}")
        elif isinstance(item, h5py.Group):
            print(f"  GROUP    {path}")
            walk_hdf5_structure(item, prefix=path, max_depth=max_depth, _depth=_depth+1)


# Open the first available VIIRS granule for 2020 (peak fire year) directly from S3
sample_year = 2020
sample_granules = granule_catalog["VNP14IMG"][sample_year]

if not sample_granules:
    print("No VNP14IMG granules found for 2020 — check CMR query.")
else:
    print(f"Opening granule 0 of {len(sample_granules)} for structure audit...")
    print(f"Granule: {sample_granules[0]['meta']['native-id']}\n")

    # earthaccess.open() returns a list of file-like objects streamed from S3
    with earthaccess.open([sample_granules[0]]) as file_handles:
        fh = file_handles[0]
        with h5py.File(fh, 'r') as hf:
            print("═══ VNP14IMG HDF5 Structure ═══")
            walk_hdf5_structure(hf)
            
            # Print global attributes
            print("\n═══ Global Attributes ═══")
            for attr_name in list(hf.attrs)[:15]:  # first 15 to keep output clean
                print(f"  {attr_name}: {hf.attrs[attr_name]}")

In [ ]:
# ── Run queries for all collections and all years ───────────────────────────
granule_catalog = {name: {} for name in COLLECTIONS}

for coll_name, coll_info in COLLECTIONS.items():

    print(f"\n── Querying {coll_name} ({coll_info['description']}) ──")

    total = 0

    for year in YEARS:

        granules = query_granules_for_year(
            short_name=coll_info["short_name"],
            version=coll_info["version"],
            year=year,
            bbox=BBOX,
            season_start=SEASON_START_MMDD,
            season_end=SEASON_END_MMDD
        )

        granule_catalog[coll_name][year] = granules

        n = len(granules)
        print(f"  {year}: {n:>5} granules")

        total += n

    print(f"  TOTAL: {total} granules across {YEARS[0]}–{YEARS[-1]}")

In [ ]:
catalog_summary = {
    coll: {str(year): len(grans) for year, grans in years.items()}
    for coll, years in granule_catalog.items()
}

with open(CATALOG_DIR / "granule_counts.json", "w") as f:
    json.dump(catalog_summary, f, indent=2)

print(f"\n✓ Catalog saved to {CATALOG_DIR / 'granule_counts.json'}")

---
## 5. VNP14IMG Variable Mapping

Based on the structure audit above (and the VNP14IMG ATBD), we define the exact
HDF5 paths for the variables we need. These paths are version-specific.

**VNP14IMG v001 key variables:**
```
/Fire Mask/fire mask        → uint8,  fire classification (7=fire, 8=fire high conf)
/Fire Radiative Power/...   → float32, FRP in MW (only at 750m; proxy at 375m)
/Brightness Temperature/... → float32, BT in K at I4 and I5
/Geolocation/latitude       → float32
/Geolocation/longitude      → float32
```

> Note: Exact paths depend on granule version. The structure audit in Cell 4 reveals
> the true paths. Update the dict below if your audit shows different paths.

In [ ]:
# ── VNP14IMG variable path mapping ────────────────────────────────────────────
# UPDATE THESE PATHS based on the structure audit output in Cell 4
VNP14_PATHS = {
    # Fire classification mask
    # Values: 0=not processed, 3=non-fire, 7=fire low conf, 8=fire nom conf, 9=fire high conf
    "fire_mask":  "Fire Mask/fire mask",

    # Brightness temperatures in Kelvin
    # These are the FireSat-analog MWIR (I4 ~3.7µm) and LWIR (I5 ~11µm) channels
    "bt_i4":      "Fire Mask/Swath_L2/brightness_temp_i4",   # adjust after audit
    "bt_i5":      "Fire Mask/Swath_L2/brightness_temp_i5",   # adjust after audit

    # Geolocation
    "latitude":   "Geolocation/latitude",
    "longitude":  "Geolocation/longitude",
    
    # Algorithm QA flags
    "algorithm_qa": "Fire Mask/algorithm QA",
}

# ── Fill values and scale factors (from VNP14IMG ATBD) ───────────────────────
VNP14_FILL = {
    "fire_mask":  0,      # 0 = not processed / fill
    "bt_i4":      -999.0,
    "bt_i5":      -999.0,
}

# Fire classes that indicate confirmed/probable fire pixels
FIRE_CLASSES_POSITIVE = {7, 8, 9}   # low, nominal, high confidence fire
FIRE_CLASSES_CANDIDATE = {3, 7, 8, 9}  # for candidate patch extraction (include non-fire for negatives)

print("✓ VNP14IMG variable paths defined")
print("  → Update VNP14_PATHS['bt_i4'] and ['bt_i5'] based on the structure audit output above")

---
## 6. Single-Granule Extraction Function

This function does the core per-granule work:
1. Opens granule from S3 file handle
2. Reads BT_I4, BT_I5, fire mask, lat, lon
3. Applies the Western US bounding box spatial filter
4. Identifies candidate thermal anomaly pixels
5. Extracts 32×32 patches centered on each candidate
6. Returns a list of patch records

In [ ]:
def extract_patches_from_viirs_granule(
    file_handle,
    granule_meta: dict,
    bbox: tuple,
    patch_size: int = 32,
    bt_threshold: float = 310.0,
    min_bt_i4_valid: float = 200.0,  # K — physical lower bound for valid BT
    max_bt_i4_valid: float = 600.0,  # K — saturation guard
    max_patches_per_granule: int = 200,
) -> list:
    """
    Extract 32×32 multispectral IR patches from a single VNP14IMG granule.

    Candidate pixels are those with BT_I4 above `bt_threshold` AND within
    the study area bounding box. For each candidate, a 32×32 patch is extracted
    in both BT_I4 and BT_I5, plus the fire mask patch for labeling.

    Parameters
    ----------
    file_handle            : S3 file-like object from earthaccess.open()
    granule_meta           : earthaccess DataGranule metadata dict
    bbox                   : (lon_min, lat_min, lon_max, lat_max)
    patch_size             : square patch side length in pixels (default 32)
    bt_threshold           : BT_I4 threshold in K for candidate selection
    min_bt_i4_valid        : lower physical validity bound for BT_I4
    max_bt_i4_valid        : upper physical validity bound for BT_I4
    max_patches_per_granule: cap to prevent single-granule memory explosion

    Returns
    -------
    list of dicts, each containing:
        granule_id  : str
        center_lat  : float
        center_lon  : float
        row_idx     : int (pixel row in original granule)
        col_idx     : int (pixel col in original granule)
        bt_i4_patch : np.ndarray (patch_size, patch_size) float32
        bt_i5_patch : np.ndarray (patch_size, patch_size) float32
        bt_diff_patch: np.ndarray (patch_size, patch_size) float32  ← BT_I4 - BT_I5
        fire_mask_patch: np.ndarray (patch_size, patch_size) uint8
        label       : int  (1=fire, 0=non-fire, based on center pixel fire_mask)
        fire_conf   : int  (fire mask class at center pixel: 0–9)
        date_str    : str  (YYYY-MM-DD)
    """
    half = patch_size // 2  # 16 pixels
    lon_min, lat_min, lon_max, lat_max = bbox
    patches = []

    with h5py.File(file_handle, 'r') as hf:

        # ── Read arrays ────────────────────────────────────────────────────────
        # Try to read BT arrays — path may vary; wrap in try/except with fallback
        try:
            lat       = hf[VNP14_PATHS["latitude"]][:].astype(np.float32)
            lon       = hf[VNP14_PATHS["longitude"]][:].astype(np.float32)
            fire_mask = hf[VNP14_PATHS["fire_mask"]][:].astype(np.uint8)
        except KeyError as e:
            print(f"  [SKIP] Missing key {e} — update VNP14_PATHS after structure audit")
            return []

        # BT arrays: try the configured path; if absent, derive from fire mask file
        # NOTE: Some VNP14IMG versions store BT in a companion VNP02IMG granule.
        # In that case, set bt_i4 = bt_i5 = None and the patch stores NaN — flagged
        # for pairing with VNP02IMG in notebook 01c.
        try:
            bt_i4 = hf[VNP14_PATHS["bt_i4"]][:].astype(np.float32)
            bt_i5 = hf[VNP14_PATHS["bt_i5"]][:].astype(np.float32)
        except KeyError:
            # BT not in this file; fill with NaN — will pair with VNP02IMG in 01c
            print("  [INFO] BT arrays not found in VNP14IMG — will pair with VNP02IMG in 01c")
            bt_i4 = np.full_like(lat, np.nan)
            bt_i5 = np.full_like(lat, np.nan)

        # ── Mask fill values ──────────────────────────────────────────────────
        bt_i4 = np.where((bt_i4 < min_bt_i4_valid) | (bt_i4 > max_bt_i4_valid), np.nan, bt_i4)
        bt_i5 = np.where((bt_i5 < min_bt_i4_valid) | (bt_i5 > max_bt_i4_valid), np.nan, bt_i5)

        # ── Spatial filter: Western US bounding box ───────────────────────────
        in_bbox = (
            (lon >= lon_min) & (lon <= lon_max) &
            (lat >= lat_min) & (lat <= lat_max)
        )

        # ── Candidate selection ───────────────────────────────────────────────
        # Adaptive threshold: use granule-local 95th percentile as a floor to avoid
        # selecting the entire desert floor in summer.
        valid_bt = bt_i4[in_bbox & np.isfinite(bt_i4)]
        if valid_bt.size < 100:
            return []  # Too few valid pixels — likely ocean or edge granule

        adaptive_threshold = max(bt_threshold, float(np.percentile(valid_bt, 97)))

        candidate_mask = (
            in_bbox &
            np.isfinite(bt_i4) &
            (bt_i4 >= adaptive_threshold)
        )

        # Also include confirmed fire pixels even if below adaptive threshold
        # (edge cases: night fires in cold ambient backgrounds)
        fire_pixel_mask = np.isin(fire_mask, list(FIRE_CLASSES_POSITIVE)) & in_bbox
        candidate_mask = candidate_mask | fire_pixel_mask

        candidate_rows, candidate_cols = np.where(candidate_mask)

        if len(candidate_rows) == 0:
            return []

        # Cap per granule to avoid memory explosion on large fire events
        if len(candidate_rows) > max_patches_per_granule:
            # Prioritize confirmed fire pixels, then highest-BT candidates
            fire_idxs = [i for i in range(len(candidate_rows))
                         if fire_mask[candidate_rows[i], candidate_cols[i]] in FIRE_CLASSES_POSITIVE]
            other_idxs = [i for i in range(len(candidate_rows)) if i not in fire_idxs]
            # Sort non-fire candidates by BT descending
            other_idxs_sorted = sorted(other_idxs,
                                       key=lambda i: bt_i4[candidate_rows[i], candidate_cols[i]],
                                       reverse=True)
            keep = fire_idxs + other_idxs_sorted[:max(0, max_patches_per_granule - len(fire_idxs))]
            candidate_rows = candidate_rows[keep]
            candidate_cols = candidate_cols[keep]

        # ── Extract granule date from metadata ────────────────────────────────
        try:
            time_str = granule_meta["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
            date_str = time_str[:10]  # 'YYYY-MM-DD'
        except (KeyError, TypeError):
            date_str = "unknown"

        granule_id = str(granule_meta.get("meta", {}).get("native-id", "unknown"))
        nrows, ncols = lat.shape

        # ── Extract patches ───────────────────────────────────────────────────
        for r, c in zip(candidate_rows, candidate_cols):
            # Boundary check: skip candidates too close to granule edge
            if r < half or r >= nrows - half or c < half or c >= ncols - half:
                continue

            r0, r1 = r - half, r + half
            c0, c1 = c - half, c + half

            bt_i4_patch   = bt_i4[r0:r1, c0:c1].astype(np.float32)
            bt_i5_patch   = bt_i5[r0:r1, c0:c1].astype(np.float32)
            bt_diff_patch = bt_i4_patch - bt_i5_patch   # core fire signal
            fm_patch      = fire_mask[r0:r1, c0:c1].astype(np.uint8)

            # Label: center pixel fire_mask value
            center_fm  = int(fire_mask[r, c])
            label      = 1 if center_fm in FIRE_CLASSES_POSITIVE else 0

            patches.append({
                "granule_id":     granule_id,
                "center_lat":     float(lat[r, c]),
                "center_lon":     float(lon[r, c]),
                "row_idx":        int(r),
                "col_idx":        int(c),
                "bt_i4_patch":    bt_i4_patch,
                "bt_i5_patch":    bt_i5_patch,
                "bt_diff_patch":  bt_diff_patch,
                "fire_mask_patch": fm_patch,
                "label":          label,
                "fire_conf":      center_fm,
                "date_str":       date_str,
            })

    return patches


print("✓ extract_patches_from_viirs_granule() defined")

---
## 7. Dry Run — Single Granule (Validation)

Before running the full 2018–2022 batch, we process one granule and inspect
the output patches to verify shapes, value ranges, and labeling logic.

In [ ]:
# ── Patch extraction parameters ──────────────────────────────────────────

PATCH_SIZE_KM = 10      # size of fire analysis patch
STRIDE_KM = 5           # overlap between patches
BT_I4_THRESHOLD = 330   # brightness temperature threshold (Kelvin)

In [ ]:
# ── Process the first 2020 VIIRS granule ──────────────────────────────────────
test_granule = granule_catalog["VNP14IMG"][2020][0]
print(f"Testing on: {test_granule['meta']['native-id']}")

fhs = earthaccess.open([test_granule])

test_patches = extract_patches_from_viirs_granule(
    file_handle  = fhs[0],
    granule_meta = test_granule,
    patch_km     = PATCH_SIZE_KM,
    stride_km    = STRIDE_KM,
    bt_threshold = BT_I4_THRESHOLD,
)

print(f"\n─── Dry run results ───")
print(f"Patches extracted: {len(test_patches)}")

if test_patches:
    fire_patches    = [p for p in test_patches if p['label'] == 1]
    nonfire_patches = [p for p in test_patches if p['label'] == 0]
    print(f"  Fire patches:     {len(fire_patches)}")
    print(f"  Non-fire patches: {len(nonfire_patches)}")

    p0 = test_patches[0]
    print(f"\nPatch 0 metadata:")
    print(f"  Center lat/lon:  ({p0['center_lat']:.3f}, {p0['center_lon']:.3f})")
    print(f"  Date:            {p0['date_str']}")
    print(f"  Label:           {p0['label']} (fire_conf={p0['fire_conf']})")
    print(f"  bt_i4_patch shape: {p0['bt_i4_patch'].shape}")
    print(f"  bt_i4 range:     {np.nanmin(p0['bt_i4_patch']):.1f} – {np.nanmax(p0['bt_i4_patch']):.1f} K")
    print(f"  bt_diff range:   {np.nanmin(p0['bt_diff_patch']):.1f} – {np.nanmax(p0['bt_diff_patch']):.1f} K")

In [ ]:
test_granule = granule_catalog["VNP14IMG"][2020][0]

print(f"Testing on: {test_granule['meta']['native-id']}")

fhs = earthaccess.open([test_granule])

test_patches = extract_patches_from_viirs_granule(
    file_handle=fhs[0],
    granule_meta=test_granule,
    bbox=BBOX,
    patch_size=32,
    bt_threshold=310.0
)

print("\n─── Dry run results ───")
print(f"Patches extracted: {len(test_patches)}")

In [ ]:
import xarray as xr

fhs = earthaccess.open([test_granule])

ds = xr.open_dataset(fhs[0], group=None)

print(ds)

In [ ]:
import numpy as np

fire_mask = ds["fire mask"].values

fire_pixels = np.argwhere(fire_mask >= 8)

print("Fire pixels detected:", len(fire_pixels))

In [ ]:
import numpy as np
import xarray as xr

fhs = earthaccess.open([test_granule])
ds = xr.open_dataset(fhs[0])

fire_mask = ds["fire mask"].values
rows = ds["FP_line"].values
cols = ds["FP_sample"].values

PATCH = 32
half = PATCH // 2

patches = []

for r, c in zip(rows, cols):

    if r-half < 0 or c-half < 0:
        continue
    if r+half >= fire_mask.shape[0] or c+half >= fire_mask.shape[1]:
        continue

    patch = fire_mask[r-half:r+half, c-half:c+half]

    if patch.shape == (PATCH, PATCH):
        patches.append(patch)

print("Extracted patches:", len(patches))

In [ ]:
extract_patches_from_viirs_granule?

In [ ]:
# ── Visualize first 8 patches ─────────────────────────────────────────────────
if len(test_patches) >= 2:
    n_show = min(8, len(test_patches))
    fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2.4, 7))
    
    for i in range(n_show):
        p = test_patches[i]
        label_str = f"FIRE\n({p['date_str']})" if p['label'] == 1 else f"non-fire\n({p['date_str']})"
        
        # Row 0: BT_I4 (MWIR — fire-sensitive)
        im0 = axes[0, i].imshow(p['bt_i4_patch'], cmap='inferno',
                                 vmin=280, vmax=400, interpolation='nearest')
        axes[0, i].set_title(label_str, fontsize=7)
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('BT I4 (3.7µm)', fontsize=8)
        
        # Row 1: BT_I5 (LWIR — surface temperature)
        axes[1, i].imshow(p['bt_i5_patch'], cmap='RdYlBu_r',
                          vmin=260, vmax=320, interpolation='nearest')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('BT I5 (11µm)', fontsize=8)
        
        # Row 2: BT differential (fire signal)
        axes[2, i].imshow(p['bt_diff_patch'], cmap='hot',
                          vmin=-5, vmax=60, interpolation='nearest')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_ylabel('ΔBT (I4−I5)', fontsize=8)
    
    fig.suptitle('FireSight-IR | Module 1a — Sample VIIRS patches\n'
                 'BT_I4 (MWIR) | BT_I5 (LWIR) | ΔBT differential',
                 fontsize=9, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01a_sample_patches.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to {FIGURES_DIR / '01a_sample_patches.png'}")

---
## 8. HDF5 Archive Writer

The patch archive is a structured HDF5 file with groups per year.
Schema:

```
firesight_viirs_patches.h5
├── year=2018/
│   ├── bt_i4        (N, 32, 32) float32
│   ├── bt_i5        (N, 32, 32) float32
│   ├── bt_diff      (N, 32, 32) float32
│   ├── fire_mask    (N, 32, 32) uint8
│   ├── labels       (N,)        uint8
│   ├── fire_conf    (N,)        uint8
│   ├── center_lat   (N,)        float32
│   ├── center_lon   (N,)        float32
│   └── date_str     (N,)        bytes (UTF-8 encoded)
├── year=2019/
│   └── ...
...
```

Chunking: `(256, 32, 32)` for spatial arrays — aligns with typical batch sizes.

In [ ]:
def write_patches_to_hdf5(
    patches: list,
    h5_path: Path,
    year: int,
    mode: str = 'a',          # 'a' = append (create if not exists)
    chunk_size: int = 256,
) -> int:
    """
    Write a list of patch dicts to an HDF5 archive, appending to the year group.
    Creates chunked, resizable datasets on first write.
    
    Returns the number of patches written.
    """
    if not patches:
        return 0

    n = len(patches)
    group_name = f"year={year}"

    # Stack arrays
    bt_i4    = np.stack([p["bt_i4_patch"]    for p in patches], axis=0)  # (n, 32, 32)
    bt_i5    = np.stack([p["bt_i5_patch"]    for p in patches], axis=0)
    bt_diff  = np.stack([p["bt_diff_patch"]  for p in patches], axis=0)
    fm       = np.stack([p["fire_mask_patch"] for p in patches], axis=0)
    labels   = np.array([p["label"]          for p in patches], dtype=np.uint8)
    conf     = np.array([p["fire_conf"]      for p in patches], dtype=np.uint8)
    clat     = np.array([p["center_lat"]     for p in patches], dtype=np.float32)
    clon     = np.array([p["center_lon"]     for p in patches], dtype=np.float32)
    dates    = np.array([p["date_str"].encode('utf-8') for p in patches])

    with h5py.File(h5_path, mode) as hf:
        if group_name not in hf:
            grp = hf.create_group(group_name)
            # Create resizable datasets with compression
            ps = PATCH_SIZE
            grp.create_dataset("bt_i4",     data=bt_i4,   maxshape=(None, ps, ps),
                               chunks=(chunk_size, ps, ps), compression="gzip", compression_opts=4)
            grp.create_dataset("bt_i5",     data=bt_i5,   maxshape=(None, ps, ps),
                               chunks=(chunk_size, ps, ps), compression="gzip", compression_opts=4)
            grp.create_dataset("bt_diff",   data=bt_diff, maxshape=(None, ps, ps),
                               chunks=(chunk_size, ps, ps), compression="gzip", compression_opts=4)
            grp.create_dataset("fire_mask", data=fm,      maxshape=(None, ps, ps),
                               chunks=(chunk_size, ps, ps), compression="gzip", compression_opts=4)
            grp.create_dataset("labels",    data=labels,  maxshape=(None,),
                               chunks=(chunk_size,))
            grp.create_dataset("fire_conf", data=conf,    maxshape=(None,),
                               chunks=(chunk_size,))
            grp.create_dataset("center_lat", data=clat,   maxshape=(None,),
                               chunks=(chunk_size,))
            grp.create_dataset("center_lon", data=clon,   maxshape=(None,),
                               chunks=(chunk_size,))
            grp.create_dataset("date_str",  data=dates,   maxshape=(None,),
                               chunks=(chunk_size,), dtype=h5py.string_dtype())
        else:
            # Append to existing datasets
            grp = hf[group_name]
            for dsname, arr in [("bt_i4", bt_i4), ("bt_i5", bt_i5), ("bt_diff", bt_diff),
                                ("fire_mask", fm)]:
                ds = grp[dsname]
                old_n = ds.shape[0]
                ds.resize(old_n + n, axis=0)
                ds[old_n:] = arr
            for dsname, arr in [("labels", labels), ("fire_conf", conf),
                                ("center_lat", clat), ("center_lon", clon), ("date_str", dates)]:
                ds = grp[dsname]
                old_n = ds.shape[0]
                ds.resize(old_n + n, axis=0)
                ds[old_n:] = arr

    return n


print("✓ write_patches_to_hdf5() defined")

---
## 9. Full Batch Processing — 2018 to 2022

This cell processes all VIIRS VNP14IMG granules for the Western US across the
2018–2022 fire seasons. Each year is processed independently so you can
checkpoint mid-run by commenting out completed years.

**Runtime estimate:** ~6–14 min per year depending on S3 network speed and
granule count. 2020 will be the largest (exceptional fire year).

**Memory:** Peak ~1–2 GB RAM (one granule in memory at a time, never accumulates).

In [ ]:
ARCHIVE_PATH = ARCHIVE_DIR / "firesight_viirs_patches.h5"

# ── Processing log ────────────────────────────────────────────────────────────
processing_log = []

for year in YEARS:
    year_granules = granule_catalog["VNP14IMG"][year]
    n_granules = len(year_granules)
    year_patches_total = 0
    year_fire_total = 0
    year_errors = 0

    print(f"\n{'═'*60}")
    print(f"Processing year {year} | {n_granules} granules")
    print(f"{'═'*60}")

    # Process granules in batches to manage S3 connection overhead
    BATCH_SIZE = 20  # open 20 granules at a time
    granule_batches = [
        year_granules[i:i + BATCH_SIZE]
        for i in range(0, n_granules, BATCH_SIZE)
    ]

    for batch_idx, batch in enumerate(tqdm(granule_batches,
                                            desc=f"{year} batch progress",
                                            unit="batch")):
        try:
            with earthaccess.open(batch) as file_handles:
                for granule, fh in zip(batch, file_handles):
                    try:
                        patches = extract_patches_from_viirs_granule(
                            file_handle   = fh,
                            granule_meta  = granule,
                            bbox          = BBOX,
                            patch_size    = PATCH_SIZE,
                            bt_threshold  = BT_I4_THRESHOLD,
                        )
                        if patches:
                            n_written = write_patches_to_hdf5(patches, ARCHIVE_PATH, year)
                            year_patches_total += n_written
                            year_fire_total += sum(p['label'] for p in patches)

                    except Exception as granule_err:
                        year_errors += 1
                        # Continue on single granule failure — don't abort year
                        if year_errors <= 5:  # Only print first 5 errors
                            print(f"  [WARN] Granule error: {granule_err}")

        except Exception as batch_err:
            print(f"  [ERROR] Batch {batch_idx} failed: {batch_err}")
            year_errors += 1
            continue

    year_nonfire = year_patches_total - year_fire_total
    print(f"  ✓ {year}: {year_patches_total:,} patches written "
          f"({year_fire_total:,} fire | {year_nonfire:,} non-fire) "
          f"| {year_errors} errors")

    processing_log.append({
        "year": year,
        "n_granules": n_granules,
        "n_patches": year_patches_total,
        "n_fire": year_fire_total,
        "n_nonfire": year_nonfire,
        "n_errors": year_errors,
    })

print(f"\n{'═'*60}")
print("BATCH COMPLETE")
print(f"Archive: {ARCHIVE_PATH}")
print(f"{'═'*60}")

# Save processing log
log_df = pd.DataFrame(processing_log)
log_df.to_csv(CATALOG_DIR / "01a_processing_log.csv", index=False)
print(log_df.to_string(index=False))

---
## 10. Archive QA — Structure, Balance, and Coverage

In [ ]:
# ── Inspect the completed archive ─────────────────────────────────────────────
print(f"Archive: {ARCHIVE_PATH}")
print(f"File size: {ARCHIVE_PATH.stat().st_size / 1e9:.2f} GB")

stats_rows = []

with h5py.File(ARCHIVE_PATH, 'r') as hf:
    for yr in YEARS:
        grp_name = f"year={yr}"
        if grp_name not in hf:
            print(f"  [MISSING] {grp_name}")
            continue
        grp    = hf[grp_name]
        n      = grp["labels"].shape[0]
        n_fire = int(grp["labels"][:].sum())
        bt_i4_sample = grp["bt_i4"][:min(500, n)]

        print(f"  {grp_name}: {n:>8,} patches | fire={n_fire:>7,} "
              f"({100*n_fire/max(n,1):.1f}%) | "
              f"BT_I4 mean={np.nanmean(bt_i4_sample):.1f}K")
        stats_rows.append({"year": yr, "n_total": n, "n_fire": n_fire,
                           "pct_fire": 100*n_fire/max(n,1)})

stats_df = pd.DataFrame(stats_rows)
print(f"\nTotal patches: {stats_df['n_total'].sum():,}")
print(f"Total fire:    {stats_df['n_fire'].sum():,}")
print(f"Overall fire%: {100*stats_df['n_fire'].sum()/max(stats_df['n_total'].sum(),1):.1f}%")

In [ ]:
# ── Class balance and annual coverage chart ────────────────────────────────────
if not stats_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Left: patch counts by year, stacked fire vs non-fire
    years_x   = stats_df["year"]
    fire_n    = stats_df["n_fire"]
    nonfire_n = stats_df["n_total"] - stats_df["n_fire"]

    axes[0].bar(years_x, fire_n,    label="Fire",     color="#E24B4A", alpha=0.85)
    axes[0].bar(years_x, nonfire_n, bottom=fire_n,    label="Non-fire", color="#378ADD", alpha=0.7)
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Patch count")
    axes[0].set_title("Patch counts by year")
    axes[0].legend()
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

    # Right: fire fraction by year
    axes[1].bar(years_x, stats_df["pct_fire"], color="#E24B4A", alpha=0.85)
    axes[1].axhline(stats_df["pct_fire"].mean(), color='k', linestyle='--', lw=1,
                    label=f"Mean: {stats_df['pct_fire'].mean():.1f}%")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Fire patches (%)")
    axes[1].set_title("Fire class fraction by year")
    axes[1].legend()

    fig.suptitle("FireSight-IR | Module 1a — Patch archive QA", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "01a_patch_qa.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to {FIGURES_DIR / '01a_patch_qa.png'}")

In [ ]:
# ── Spatial coverage check — scatter plot of patch center locations ────────────
import random

all_lats, all_lons, all_labels = [], [], []

with h5py.File(ARCHIVE_PATH, 'r') as hf:
    for yr in YEARS:
        grp_name = f"year={yr}"
        if grp_name not in hf:
            continue
        grp = hf[grp_name]
        n   = grp["labels"].shape[0]
        # Sample up to 2000 patches per year for the scatter plot
        idx = sorted(random.sample(range(n), min(2000, n)))
        all_lats.extend(grp["center_lat"][idx].tolist())
        all_lons.extend(grp["center_lon"][idx].tolist())
        all_labels.extend(grp["labels"][idx].tolist())

if all_lats:
    fig, ax = plt.subplots(figsize=(7, 6))
    colors = ['#378ADD' if l == 0 else '#E24B4A' for l in all_labels]
    ax.scatter(all_lons, all_lats, c=colors, s=2, alpha=0.4, linewidths=0)

    # Western US state boundaries (rough approximate)
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((-124.5, 32.5), 10.5, 14.0, fill=False,
                            edgecolor='gray', linewidth=1.0, linestyle='--'))

    ax.set_xlim(-125, -113.5)
    ax.set_ylim(32, 47)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("FireSight-IR | Patch spatial coverage (sampled)\n"
                 "Red = fire  |  Blue = non-fire")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "01a_spatial_coverage.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to {FIGURES_DIR / '01a_spatial_coverage.png'}")

---
## 11. Summary and Next Steps

### What was built
- Authenticated to NASA Earthdata S3 — zero local downloads
- Queried CMR for VNP14IMG granules, Western US, fire season 2018–2022
- Extracted 32×32 BT_I4/BT_I5/ΔBT patches around thermal anomaly candidates
- Written structured, compressed, resizable HDF5 archive with per-year groups
- Verified spatial coverage, class balance, and BT value ranges

### Known limitations and follow-up flags
1. **BT path update** — if the structure audit in Cell 4 reveals different HDF5 paths
   for BT_I4/I5, update `VNP14_PATHS` and re-run the batch (HDF5 is appendable).
2. **VNP02IMG pairing** — if BT arrays were absent in VNP14IMG, notebook `01c` will
   pair with the companion VNP02IMG (calibrated radiance) product.
3. **MODIS pairing** — the current archive is VIIRS-only. Notebook `01c` adds MODIS
   MOD14/MYD14 co-registered onto the same 375 m grid.
4. **Class imbalance** — if fire patches < 5% of total, apply stratified sampling
   or weighted loss in Module 3 (document ratio here for Module 2).

### Outputs from this notebook
| Output | Path |
|---|---|
| VIIRS patch archive | `data/patches/firesight_viirs_patches.h5` |
| Granule count catalog | `data/catalogs/granule_counts.json` |
| Processing log | `data/catalogs/01a_processing_log.csv` |
| Sample patch figure | `figures/01a_sample_patches.png` |
| QA figure | `figures/01a_patch_qa.png` |
| Spatial coverage | `figures/01a_spatial_coverage.png` |

### Next notebook
**`01b_era5_merra2_download.ipynb`** — Authenticate to Copernicus CDS, query ERA5
atmospheric profiles (T, q, PBL height) over the Western US for 2018–2022 fire season,
and write per-day xarray datasets aligned to the patch archive locations.